# 7B vs 125M on CORD-v2 - GPU steps

Everything that needs a GPU. The harness checks run first and on purpose: if the
converter ceiling is not exactly 1.000, no number produced later in this notebook is
worth recording.

Runtime: **A100 High-RAM**. Roughly 2-3 hours end to end.

In [ ]:
!git clone https://github.com/Perlious-Savage/qwen-qlora-vs-layoutlmv3.git
%cd qwen-qlora-vs-layoutlmv3

In [ ]:
!pip install -q -r requirements-train.txt

## 0. Verify the harness before spending any GPU time

Three checks, in order of how badly they invalidate the result:

1. the scoring code is byte-identical to Project 1's
2. the JSON-to-spans converter loses nothing (ceiling exactly 1.000)
3. no test document also appears in train

In [ ]:
!sha256sum -c VENDORED.sha256
!pytest tests/ -q

In [ ]:
!python roundtrip_check.py

In [ ]:
!python scripts/check_splits.py

Set the generation token cap from the gold completion lengths. A completion cut off
mid-JSON is unparseable and scores zero, which reads as a model failure rather than the
config failure it is. This fails loudly if `MAX_NEW_TOKENS` is too low.

In [ ]:
!python -m src.eval_llm --lengths

## 1. Base model, few-shot

The number the fine-tune has to beat. Exemplars come from the training split only.
Watch `parse_rate` as much as F1 - a base model usually fails the output contract long
before it fails at reading receipts.

In [ ]:
!python -m src.eval_llm --config base --fewshot 2

## 2. QLoRA fine-tune

Smoke test first - confirms the whole path works before spending a real run on it.

In [ ]:
!python -m src.train_qlora --max-train 40 --epochs 1 --output outputs/smoke

In [ ]:
!python -m src.train_qlora --epochs 3

## 3. Score the fine-tune on the same test split, with the same decoding config

In [ ]:
!python -m src.eval_llm --config qlora --adapter outputs/qwen-cord-lora

## 4. Benchmark

All configurations in one process on one GPU so the rows are comparable. The LayoutLMv3
row is a different workload - one forward pass, not generation - and the table says so.

In [ ]:
!python bench.py --adapter outputs/qwen-cord-lora

In [ ]:
!pip install -q -r requirements-serve.txt
!python bench.py --adapter outputs/qwen-cord-lora --engine vllm --skip-encoder

## 5. Optional: LoRA rank sweep

Adds roughly three hours. Skip it unless the sensitivity question is worth that.

In [ ]:
# !python sweep.py --fast

## 6. The comparison

Tables are generated from the artifacts, never typed. If the QLoRA-vs-LayoutLMv3 gap
comes out narrower than 0.05, rerun the fine-tune at seeds 1 and 2 and report mean and
spread instead of a point estimate - at that margin a single seed cannot support either
conclusion.

In [ ]:
!python compare.py

## 7. Commit the measured results back to the repo

The artifacts are the evidence. They belong in git, not only in this runtime.

In [ ]:
import os
os.environ['GH_TOKEN'] = ''  # paste a token with repo scope, or push from the local machine
!git config user.name "Perlious-Savage"
!git add -f artifacts/
!git commit -q -m "Add measured results: base, QLoRA, benchmark"
# !git push